# Avance Proyecto 01
##### Análisis Numérico de la Evolución del Precio de Bitcoin (BTC/USDT)

 **Integrantes:** Nicolás Yugsi, Sebastián Bravo, Erick Toapanta, Heidy Tenelema.


## FASE 1: PREPROCESAMIENTO, ARITMÉTICA Y TIEMPO

1. **Carga y Limpieza:** Se importan los primeros 10,000 datos del dataset.
2. **Aritmética de Computador:** Se convierten los precios crudos (enteros escalados) a formato de punto flotante (`float64`), asegurando una precisión de 8 decimales para minimizar el error de redondeo en los cálculos posteriores.
3. **Linealización del Tiempo:** Se transforma la fecha `open_time` en un índice numérico continuo $t = 0, 1, 2...$ para habilitar el uso de funciones matemáticas dependientes de $x$.

In [17]:
import pandas as pd
import numpy as np

# Configuración para que TODOS los números se vean con 8 decimales
pd.options.display.float_format = '{:.8f}'.format

# --- DEFINICIÓN DE FUNCIONES ---

def cargar_dataset(nombre_archivo):
    # Cargamos el archivo especificando el separador de punto y coma
    print("1. Cargando archivo CSV...")
    df = pd.read_csv(nombre_archivo, sep=';')
    return df

def limpiar_datos(df):
    print("2. Procesando datos y convirtiendo variables...")

    # Seleccionamos las columnas que vamos a utilizar
    cols = ['open_time', 'close_price', 'high_price', 'low_price', 'volume', 'EMA']
    datos = df[cols].copy()

    # A. LIMPIEZA ARITMÉTICA (Precios a Dólares)
    # Dividimos por 100 millones (10^8) para obtener el valor real
    factor_precio = 100000000.0
    datos['close_price'] = datos['close_price'] / factor_precio
    datos['high_price']  = datos['high_price'] / factor_precio
    datos['low_price']   = datos['low_price'] / factor_precio

    # B. LIMPIEZA DE LA EMA
    # Reemplazamos coma por punto y ajustamos la escala (10^13)
    datos['EMA'] = datos['EMA'].astype(str).str.replace(',', '.')
    datos['EMA'] = datos['EMA'].astype(float) / 10000000000000.0

    # C. CONVERSIÓN DE TIEMPO (De Fecha a Número)
    # Convertimos la columna 'open_time' a formato de fecha real
    datos['open_time'] = pd.to_datetime(datos['open_time'])

    # Creamos la variable numérica 'x_tiempo' usando el Timestamp
    # Esto convierte la fecha en segundos (un número lineal único para cada dato)
    datos['x_tiempo'] = datos['open_time'].apply(lambda x: x.timestamp())

    return datos

# --- EJECUCIÓN DEL PROCESO ---

# 1. Cargar
df_crudo = cargar_dataset('CRYPTO_BTC_USD.csv')

# 2. Limpiar
df_final = limpiar_datos(df_crudo)

# 3. Mostrar resultado
print("3. Proceso finalizado.")
print("")
print("Tabla de datos con los primeros 5 registros")

# Mostramos la tabla. Incluyendo x_tiempo
print("")
df_final.head()

1. Cargando archivo CSV...
2. Procesando datos y convirtiendo variables...
3. Proceso finalizado.

Tabla de datos con los primeros 5 registros



,open_time,close_price,high_price,low_price,volume,EMA,x_tiempo
0,2017-08-18 13:00:00+00:00,4293.09000000,4318.16000000,4221.05000000,4653376700.00000000,4312.08850350,1503061200.00000000
1,2017-08-18 14:00:00+00:00,4259.40000000,4293.09000000,4193.70000000,7436894300.00000000,430.39825799,1503064800.00000000
2,2017-08-18 15:00:00+00:00,4236.89000000,4259.40000000,4200.00000000,3994771700.00000000,429.36606445,1503068400.00000000
3,2017-08-18 16:00:00+00:00,4250.34000000,4283.79000000,4234.54000000,4503882400.00000000,4.28699593,1503072000.00000000
4,2017-08-18 17:00:00+00:00,4193.35000000,4250.34000000,4066.53000000,6269146600.00000000,427.25888638,1503075600.00000000


## FASE 2: INTERPOLACIÓN (SUAVIZADO DE CURVA)

Aplicamos el método de **Splines Cúbicos** sobre los datos procesados anteriormente para generar una función continua $S(t)$ que conecte todos los precios de cierre sin generar picos angulares (no derivables), simulando el comportamiento fluido del mercado.

## FASE 3 (A): TENDENCIA POR MÍNIMOS CUADRADOS

Se aplica una regresión polinomial (Mínimos Cuadrados) para modelar el comportamiento general del activo y obtener la ecuación matemática de la tendencia ($y = mx + b$) filtrando el ruido de corto plazo interpolado en la fase anterior.

## FASE 3 (B): VALIDACIÓN Y ANÁLISIS

Evaluamos la fiabilidad del modelo matemático mediante dos enfoques:
1. **Bandas de Error (Volatilidad):** Se grafican los precios Máximo (`High`) y Mínimo (`Low`) para visualizar el rango de incertidumbre en cada punto interpolado.
2. **Validación Cruzada:** Se compara la tendencia calculada (Mínimos Cuadrados) contra el indicador técnico EMA (Media Móvil Exponencial) para corroborar la dirección del mercado.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def calcular_tendencia
